# 1D CNN over the sensor array

Every model tried so far (RF, LightGBM, XGBoost, SVM, MLP, DANN, KD, VBS) treats `feat_1..feat_128`
as a **flat 128-vector**, which throws away a fact we actually know about the data:

> The layout is **sensor-major**: 16 sensors x 8 descriptors, stride 8.
> `feat_1, feat_9, ..., feat_121` are the 16 steady-state dR columns; the other 7 per block are
> smaller EMA transients.

This notebook uses that structure directly.

### Architecture

Input is reshaped to `(8 descriptors, 16 sensors)` and treated as an 8-channel signal of length 16:

1. **`Conv1d(8 -> width, kernel_size=1)`** — the same learned function applied to **every sensor
   independently**. This is the real inductive bias: 16 sensors exposed to the same gas should be
   read by the same descriptor-processing function, so their weights are shared instead of the
   model learning 16 unrelated copies.
2. **`Conv1d(width, width, kernel_size=3)`** — mixes *neighbouring* sensors (in this array sensors
   are grouped by type, so adjacency is not arbitrary).
3. **Mean + max pooling across the sensor axis** — a fixed-size, roughly permutation-invariant
   summary of the array, rather than a position-dependent flattening.
4. Concentration (raw + log) is concatenated at the head, not fed through the conv stack, since it
   is not a sensor reading.

### Two things carried over from the main notebook

- **Forward-chaining CV only** — train on batches `< n`, validate on `n`. Never trains on the future.
- **Scored on the large-drift folds as well as the mean.** Selecting on the CV mean is exactly what
  misled the OSC k=2 choice: the two most recent folds had tiny drift steps (1.42, 1.44) and
  inflated the average, while the real batch-9 -> 10 jump is 7.22.

A signed log transform is applied before standardising, because the raw magnitudes span five orders
of magnitude (`feat_1` ~ 1e5 vs the transients ~ 1e0) and neural nets handle that badly.

In [1]:
import time
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import f1_score, accuracy_score
from sklearn.preprocessing import StandardScaler

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"device: {DEVICE}"
      + (f"  ({torch.cuda.get_device_name(0)}, sm_{'%d%d' % torch.cuda.get_device_capability(0)})"
         if torch.cuda.is_available() else "  [WARNING: no GPU]"))

train = pd.read_csv("data/train.csv")
test = pd.read_csv("data/test.csv")
sample_sub = pd.read_csv("data/sample_submission.csv")

FEAT = [f"feat_{i}" for i in range(1, 129)]
EXTRA = ["concentration"]
N_SENSORS, N_DESC = 16, 8
CLASSES = sorted(train["gas_class"].unique())

# Verify the sensor-major assumption before relying on it: reshaped to (16, 8), descriptor 0
# must be the huge steady-state dR column for every sensor.
_blk = train[FEAT].values.reshape(-1, N_SENSORS, N_DESC)
_mag = np.abs(_blk).mean(axis=0)          # (16 sensors, 8 descriptors)
assert (_mag[:, 0] > 10 * _mag[:, 1:].max(axis=1)).all(), "layout is not sensor-major stride 8"
print(f"\nlayout verified: (16 sensors x 8 descriptors), descriptor 0 is the steady-state dR")
print(f"  mean |descriptor 0| across sensors: {_mag[:, 0].mean():,.0f}")
print(f"  mean |descriptors 1-7|:             {_mag[:, 1:].mean():,.1f}")
print(f"\ntrain {train.shape}, test {test.shape}, classes {CLASSES}")

device: cuda  (NVIDIA GeForce RTX 5060 Ti, sm_120)

layout verified: (16 sensors x 8 descriptors), descriptor 0 is the steady-state dR
  mean |descriptor 0| across sensors: 23,218
  mean |descriptors 1-7|:             10.5

train (10310, 132), test (3600, 131), classes [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6)]


In [2]:
def signed_log(a):
    """Raw magnitudes span ~5 orders of magnitude; compress them but keep the sign."""
    return np.sign(a) * np.log1p(np.abs(a))


def prepare(fit_df, *apply_dfs):
    """Fit scalers on fit_df only (no leakage from the validation/test batch), then transform.
    Returns, per frame: X_grid (N, 8, 16) for the conv stack and X_extra (N, 2) for the head."""
    fs = StandardScaler().fit(signed_log(fit_df[FEAT].values))
    es = StandardScaler().fit(np.column_stack([fit_df[EXTRA].values,
                                               np.log1p(fit_df["concentration"].values)]))
    out = []
    for df in (fit_df,) + apply_dfs:
        g = fs.transform(signed_log(df[FEAT].values))
        g = g.reshape(-1, N_SENSORS, N_DESC).transpose(0, 2, 1)   # -> (N, 8 desc, 16 sensors)
        e = es.transform(np.column_stack([df[EXTRA].values,
                                          np.log1p(df["concentration"].values)]))
        out.append((g.astype(np.float32), e.astype(np.float32)))
    return out


_g, _e = prepare(train)[0]
print(f"conv input {_g.shape}  (N, descriptors, sensors)   head extras {_e.shape}")

conv input (10310, 8, 16)  (N, descriptors, sensors)   head extras (10310, 2)


In [3]:
class SensorCNN(nn.Module):
    """kernel_size=1 over the sensor axis == one shared per-sensor function (weight sharing
    across the 16 sensors), then a width-3 conv to mix neighbouring sensors, then pooling
    across sensors so the array summary does not depend on sensor position."""

    def __init__(self, n_desc=N_DESC, n_extra=2, n_classes=6, width=64, p_drop=0.3):
        super().__init__()
        self.per_sensor = nn.Sequential(
            nn.Conv1d(n_desc, width, kernel_size=1), nn.BatchNorm1d(width), nn.ReLU(),
            nn.Conv1d(width, width, kernel_size=1), nn.BatchNorm1d(width), nn.ReLU(),
        )
        self.across_sensors = nn.Sequential(
            nn.Conv1d(width, width, kernel_size=3, padding=1), nn.BatchNorm1d(width), nn.ReLU(),
        )
        self.head = nn.Sequential(
            nn.Linear(width * 2 + n_extra, 128), nn.ReLU(), nn.Dropout(p_drop),
            nn.Linear(128, n_classes),
        )

    def forward(self, grid, extra):
        h = self.across_sensors(self.per_sensor(grid))
        pooled = torch.cat([h.mean(dim=-1), h.amax(dim=-1), extra], dim=1)  # pool over sensors
        return self.head(pooled)


class FlatMLP(nn.Module):
    """Control: identical inputs, but flattened -- no sensor structure. Roughly matched capacity,
    so any gap is attributable to the structural prior rather than to parameter count."""

    def __init__(self, n_in=N_SENSORS * N_DESC + 2, n_classes=6, width=256, p_drop=0.3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_in, width), nn.BatchNorm1d(width), nn.ReLU(), nn.Dropout(p_drop),
            nn.Linear(width, 128), nn.BatchNorm1d(128), nn.ReLU(), nn.Dropout(p_drop),
            nn.Linear(128, n_classes),
        )

    def forward(self, grid, extra):
        return self.net(torch.cat([grid.flatten(1), extra], dim=1))


for _m in (SensorCNN(), FlatMLP()):
    print(f"{_m.__class__.__name__:<10} params: {sum(p.numel() for p in _m.parameters()):,}")

SensorCNN  params: 35,014
FlatMLP    params: 67,974


In [4]:
def fit_net(model_cls, tr_pack, y_tr, epochs=80, bs=256, lr=2e-3, wd=1e-4,
            class_weight=False, seed=0):
    torch.manual_seed(seed)
    g, e = tr_pack
    model = model_cls().to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)

    G = torch.tensor(g, device=DEVICE)
    E = torch.tensor(e, device=DEVICE)
    Y = torch.tensor(y_tr, dtype=torch.long, device=DEVICE)

    w = None
    if class_weight:
        cnt = torch.bincount(Y, minlength=len(CLASSES)).float()
        w = (cnt.sum() / (len(CLASSES) * cnt.clamp(min=1))).to(DEVICE)

    for _ in range(epochs):
        model.train()
        for idx in torch.randperm(len(G), device=DEVICE).split(bs):
            if len(idx) < 2:      # BatchNorm needs >1 sample
                continue
            loss = F.cross_entropy(model(G[idx], E[idx]), Y[idx], weight=w)
            opt.zero_grad(); loss.backward(); opt.step()
        sched.step()
    model.eval()
    return model


@torch.no_grad()
def predict_net(model, pack):
    g, e = pack
    out = model(torch.tensor(g, device=DEVICE), torch.tensor(e, device=DEVICE))
    return np.array(CLASSES)[out.argmax(1).cpu().numpy()]


def forward_chaining(model_cls, seeds=(0, 1, 2), class_weight=False, label=""):
    """Train on batches < n, validate on n. Averaged over seeds because single-run NN scores
    on a 161-row fold are extremely noisy."""
    rows = []
    for vb in sorted(train["batch"].unique())[1:]:
        tr = train[train["batch"] < vb]
        va = train[train["batch"] == vb]
        tr_pack, va_pack = prepare(tr, va)
        y_tr = np.searchsorted(CLASSES, tr["gas_class"].values)
        f1s = [f1_score(va["gas_class"], predict_net(
            fit_net(model_cls, tr_pack, y_tr, class_weight=class_weight, seed=s), va_pack),
            average="macro") for s in seeds]
        rows.append(dict(batch=vb, n=len(va), f1=float(np.mean(f1s)), sd=float(np.std(f1s))))
    df = pd.DataFrame(rows)
    print(f"{label}: mean macro-F1 = {df.f1.mean():.4f}  (seed sd {df.sd.mean():.3f})")
    for _, r in df.iterrows():
        print(f"    batch {int(r.batch)} (n={int(r.n)}): {r.f1:.4f}")
    return df

In [5]:
# Drift step per fold, so results can be read against the shift each fold actually faced.
# The batch 9 -> 10 step (the one the submission must survive) is 7.22.
_sc = StandardScaler().fit(train[FEAT])
_X = _sc.transform(train[FEAT])
_cent = {b: _X[train["batch"].values == b].mean(0) for b in sorted(train["batch"].unique())}
DRIFT_STEP = {b: float(np.linalg.norm(_cent[b] - _cent[b - 1])) for b in range(2, 10)}
LARGE_DRIFT = [b for b, s in DRIFT_STEP.items() if s >= 5.0]
print("drift step per fold:", {b: round(s, 2) for b, s in DRIFT_STEP.items()})
print(f"large-drift folds (step >= 5, comparable to the 7.22 of batch 9->10): {LARGE_DRIFT}\n")

t0 = time.time()
res = {}
res["1D CNN (sensor structure)"] = forward_chaining(SensorCNN, label="1D CNN (sensor structure)")
print()
res["Flat MLP (control)"] = forward_chaining(FlatMLP, label="Flat MLP (control)")
print()
res["1D CNN + class weights"] = forward_chaining(SensorCNN, class_weight=True,
                                                 label="1D CNN + class weights")
print(f"\n[{time.time() - t0:.1f}s]")

drift step per fold: {2: 5.02, 3: 5.55, 4: 9.15, 5: 7.47, 6: 9.59, 7: 1.42, 8: 5.57, 9: 1.44}
large-drift folds (step >= 5, comparable to the 7.22 of batch 9->10): [2, 3, 4, 5, 6, 8]

1D CNN (sensor structure): mean macro-F1 = 0.8028  (seed sd 0.041)
    batch 2 (n=1244): 0.5223
    batch 3 (n=1586): 0.8841
    batch 4 (n=161): 0.8930
    batch 5 (n=197): 0.9630
    batch 6 (n=2300): 0.6470
    batch 7 (n=3613): 0.7296
    batch 8 (n=294): 0.8866
    batch 9 (n=470): 0.8971

Flat MLP (control): mean macro-F1 = 0.8390  (seed sd 0.015)
    batch 2 (n=1244): 0.6671
    batch 3 (n=1586): 0.9676
    batch 4 (n=161): 0.8455
    batch 5 (n=197): 0.9933
    batch 6 (n=2300): 0.6906
    batch 7 (n=3613): 0.8592
    batch 8 (n=294): 0.9373
    batch 9 (n=470): 0.7513

1D CNN + class weights: mean macro-F1 = 0.8030  (seed sd 0.034)
    batch 2 (n=1244): 0.5124
    batch 3 (n=1586): 0.9319
    batch 4 (n=161): 0.8603
    batch 5 (n=197): 0.9670
    batch 6 (n=2300): 0.6336
    batch 7 (n=3613): 0.

In [6]:
summary = []
for name, df in res.items():
    big = df[df.batch.isin(LARGE_DRIFT)]
    summary.append(dict(model=name, mean_f1=df.f1.mean(), large_drift_f1=big.f1.mean(),
                        worst_fold=df.f1.min()))
summary = pd.DataFrame(summary).sort_values("large_drift_f1", ascending=False)

print("=" * 88)
print("Ranked by LARGE-DRIFT folds -- the regime batch 10 actually sits in")
print("=" * 88)
print(f"{'model':<32}{'mean F1':>10}{'large-drift F1':>16}{'worst fold':>13}")
print("-" * 88)
for _, r in summary.iterrows():
    print(f"{r.model:<32}{r.mean_f1:>10.4f}{r.large_drift_f1:>16.4f}{r.worst_fold:>13.4f}")
print("-" * 88)
print("""
Reference points from the main notebook, same forward-chaining scheme:
  RF baseline          mean 0.7807
  OSC k=2              mean 0.8477   (but only 10.6% of the real batch-10 shift lies in the
                                      subspace it removes, vs 91.8% for the batch-9 fold it
                                      was validated on -- so that number does not transfer)
  Flat MLP (main nb)   mean 0.7786
""")
print("The comparison that matters here is CNN vs Flat MLP: same inputs, same budget,")
print("the only difference is whether the 16x8 sensor structure is exploited.")

Ranked by LARGE-DRIFT folds -- the regime batch 10 actually sits in
model                              mean F1  large-drift F1   worst fold
----------------------------------------------------------------------------------------
Flat MLP (control)                  0.8390          0.8502       0.6671
1D CNN (sensor structure)           0.8028          0.7993       0.5223
1D CNN + class weights              0.8030          0.7972       0.5124
----------------------------------------------------------------------------------------

Reference points from the main notebook, same forward-chaining scheme:
  RF baseline          mean 0.7807
  OSC k=2              mean 0.8477   (but only 10.6% of the real batch-10 shift lies in the
                                      subspace it removes, vs 91.8% for the batch-9 fold it
                                      was validated on -- so that number does not transfer)
  Flat MLP (main nb)   mean 0.7786

The comparison that matters here is CNN vs Flat

In [7]:
# Fit on all 9 labelled batches and predict batch 10.
# Written to data/submission_cnn.csv -- deliberately NOT overwriting data/submission.csv,
# since on these results the CNN has not earned the submission slot.
best_name = summary.iloc[0]["model"]
use_cw = "class weights" in best_name
model_cls = FlatMLP if "Flat" in best_name else SensorCNN
print(f"refitting: {best_name}")

tr_pack, te_pack = prepare(train, test)
y_all = np.searchsorted(CLASSES, train["gas_class"].values)

votes = np.zeros((len(test), len(CLASSES)))
for s in (0, 1, 2, 3, 4):          # seed ensemble: single-net predictions are noisy
    m = fit_net(model_cls, tr_pack, y_all, class_weight=use_cw, seed=s)
    with torch.no_grad():
        p = F.softmax(m(torch.tensor(te_pack[0], device=DEVICE),
                        torch.tensor(te_pack[1], device=DEVICE)), dim=1)
    votes += p.cpu().numpy()
pred = np.array(CLASSES)[votes.argmax(1)]

sub = pd.DataFrame({"measurement_id": test["measurement_id"], "gas_class": pred})
assert list(sub.columns) == list(sample_sub.columns)
assert len(sub) == len(sample_sub)
assert (sub["measurement_id"].values == sample_sub["measurement_id"].values).all()
assert sub["gas_class"].isin(range(1, 7)).all() and sub["gas_class"].notna().all()
sub.to_csv("data/submission_cnn.csv", index=False)

print(f"wrote data/submission_cnn.csv  (5-seed ensemble; data/submission.csv untouched)")
vc = sub["gas_class"].value_counts().sort_index()
print("\npredicted distribution vs the 600/class the competition states:")
for c in CLASSES:
    print(f"  class {c}: {vc.get(c, 0):>5}  ({vc.get(c, 0) - 600:+d})")
print(f"  total absolute deviation: {int((vc.reindex(CLASSES).fillna(0) - 600).abs().sum())}")

refitting: Flat MLP (control)
wrote data/submission_cnn.csv  (5-seed ensemble; data/submission.csv untouched)

predicted distribution vs the 600/class the competition states:
  class 1:   472  (-128)
  class 2:   461  (-139)
  class 3:   587  (-13)
  class 4:   516  (-84)
  class 5:   599  (-1)
  class 6:   965  (+365)
  total absolute deviation: 730
